# Data Fusion 2025 — Задача 2 "4cast"
## Прогнозирование еженедельных переводов юридических лиц

**Задача:** предсказать суммарные переводы 51 963 клиентов банка за 12 недель,  
используя историю за 105 недель (105).

**Метрика:** средний по клиентам RMSLE:
$$\overline{RMSLE} = \frac{1}{N}\sum_{i=1}^N \sqrt{\frac{1}{T}\sum_{t=1}^T (\log(1+y_{it}) - \log(1+\hat{y}_{it}))^2}$$

**Public leaderboard:** недели 118–121 (4 из 12 прогнозных).

---


## Скачивание данных

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
import warnings
import json
from pathlib import Path
import gc, time

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.4
sns.set_palette("tab10")

DATA_DIR = Path("data")
FIG_DIR  = Path("figures1")
FIG_DIR.mkdir(exist_ok=True)

# Splits (из calendar_extended.csv)
TRAIN_WEEKS    = list(range(0, 106))     # 0–105  → train
VAL_PUB_WEEKS  = list(range(106, 110))  # 106–109 → validation_public
VAL_PRIV_WEEKS = list(range(110, 118))  # 110–117 → validation_private
PUBLIC_WEEKS   = list(range(118, 122))  # 118–121 → public leaderboard
PRIVATE_WEEKS  = list(range(122, 130))  # 122–129 → private leaderboard
FORECAST_WEEKS = list(range(118, 130))  # все 12 недель прогноза

print("Config loaded.")


Config loaded.


In [5]:
target_series = pd.read_parquet(DATA_DIR / "target_series.parquet") # weeks 0–105
target_series_extended = pd.read_parquet(DATA_DIR / "target_series_extended.parquet")       # weeks 105–117

target_series = pd.concat([target_series, target_series_extended], ignore_index=True)

In [6]:
print("Loading data...")
calendar = pd.read_csv(DATA_DIR / "calendar_extended.csv", parse_dates=["date"])
sample   = pd.read_csv(DATA_DIR / "sample_submit_extended.csv")
profiles = pd.read_parquet(DATA_DIR / "profiles_extended.parquet")

target_series["week"] = target_series["week"].astype(int)

TX_FILES = sorted(DATA_DIR.glob("transactions_[1-5].parquet"))
print(f"Found {len(TX_FILES)} transaction files: {[f.name for f in TX_FILES]}")

t0 = time.time()
trns_chunks = []
for f in TX_FILES:
    print(f"Reading {f.name} ...")
    chunk = pd.read_parquet(f)
    print(f"  -> {len(chunk):,} rows")
    trns_chunks.append(chunk)

trns_raw = pd.concat(trns_chunks, ignore_index=True)
del trns_chunks
gc.collect()

Loading data...
Found 5 transaction files: ['transactions_1.parquet', 'transactions_2.parquet', 'transactions_3.parquet', 'transactions_4.parquet', 'transactions_5.parquet']
Reading transactions_1.parquet ...
  -> 52,089,892 rows
Reading transactions_2.parquet ...
  -> 55,045,806 rows
Reading transactions_3.parquet ...
  -> 59,510,350 rows
Reading transactions_4.parquet ...
  -> 61,492,247 rows
Reading transactions_5.parquet ...
  -> 30,036,951 rows


1454

## Подготовка данных 

In [7]:
trns = trns_raw.copy()

In [8]:
trns["date"] = pd.to_datetime(trns["date"], utc=True).dt.tz_localize(None).dt.normalize()

all_inn = target_series["inn_id"].unique()

cal_105 = calendar.loc[calendar["week"] <= 117].copy()
cal_105["date"] = pd.to_datetime(cal_105["date"]).dt.tz_localize(None).dt.normalize()
all_dates = np.sort(cal_105["date"].drop_duplicates().values)

mi = pd.MultiIndex.from_product([all_inn, all_dates], names=["doc_payer_inn", "date"])
new_trns = mi.to_frame(index=False)

new_trns["doc_payee_inn"] = ""
new_trns["trns_count"] = 1.0
new_trns["trns_amount"] = 0
new_trns["doc_payer_bank_name_encoded"] = 51
new_trns["doc_payee_bank_name_encoded"] = 0
new_trns["doc_payer_bank_name_flag"] = 1
new_trns["doc_payee_bank_name_flag"] = 0
new_trns["trns_class_encoded"] = 0

trns = pd.concat([trns, new_trns], ignore_index=True)

trns = pd.merge(trns, calendar, on='date', how='left')

allowed = target_series["inn_id"].unique()
trns = trns[trns["doc_payer_inn"].isin(allowed)]

In [9]:
# построение ряда с суммами переводов по неделям только со счетов втб 
trns_vtb = trns[trns["doc_payer_bank_name_flag"] == 1].copy()
trns_vtb = trns_vtb.groupby(["week", "doc_payer_inn"], as_index=False, sort=False).agg({
    "trns_amount": "sum",
    "part": "first",
}).rename(columns={
                 "doc_payer_inn": "inn_id",
                 "trns_amount": "trns_amount_only_from_vtb",
             })

trns_vtb.drop(columns=["part"], inplace=True)

In [10]:
trns = trns.groupby(["week", "doc_payer_inn"], as_index=False, sort=False).agg({
    "trns_amount": "sum",
    "part": "first",
}).rename(columns={
                 "doc_payer_inn": "inn_id",
                 "trns_amount": "trns_amount",
             })

In [11]:
trns = pd.merge(trns, trns_vtb, on=["week", "inn_id"], how="left")  


In [12]:
trns = pd.merge(trns, target_series, on=["week", "inn_id"], how="left")  

In [13]:
trns.head()

,week,inn_id,trns_amount,part,trns_amount_only_from_vtb,target
0,0,inn1000051,4.221593e+05,train,4.221593e+05,4.221593e+05
1,0,inn1000367,3.994532e+05,train,3.026673e+05,3.026673e+05
2,0,inn1000484,9.672847e+05,train,5.892931e+05,5.892931e+05
3,0,inn1000624,1.684209e+05,train,2.236366e+04,2.236366e+04
4,0,inn1000884,5.724560e+06,train,2.112144e+06,2.112144e+06


In [14]:
del trns_vtb
del new_trns
del cal_105
gc.collect()

0

## Построение признаков

### Категориальные признаки

In [15]:
from sklearn.preprocessing import OrdinalEncoder
import joblib

In [16]:
pr = profiles.copy()
pr.isna().sum()

inn_id                                0
report_date                           0
ipul                                  0
id_region                       1141970
main_okved_group                1299647
diff_datopen_report_date_flg          0
dtype: int64

In [17]:
pr['id_region'].isna().sum() / pr['id_region'].count() * 100

np.float64(3.066721528945321)

In [18]:
pr['main_okved_group'].isna().sum() / pr['main_okved_group'].count() * 100

np.float64(3.5049991762363177)

In [19]:
# todo: изобразить на графике количество уникальных значений в profiles
# цель:показать, что уникальных значений не много и что сложный кодировщик не нужен 

In [20]:
# todo: распределение записей по inn_id
# цель: показать, что inn_id не изменяются во времени


Используем OrdinalEncoder, так как он часто используется с моделью XGBoost, подходит для категориальных колонок с умеренным разбросом значений и компактно кодирует признаки. (При увеличесии выборки начались проблемы с нехваткой памяти, поэтому нужно экономить)

Пустые значения считаются отдельной категорией. 

In [21]:
allowed = target_series["inn_id"].unique()
cat_cols = ["ipul", "id_region", "main_okved_group", "diff_datopen_report_date_flg"]

prof = (
    profiles[profiles["inn_id"].isin(allowed)]
    .sort_values("report_date")
    .groupby("inn_id", as_index=False)
    .last()
)
prof[cat_cols] = prof[cat_cols].fillna("unknown").astype("string")

enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=np.int32,
)
prof[cat_cols] = enc.fit_transform(prof[cat_cols]).astype(np.int32)

joblib.dump(enc, DATA_DIR / "ordinal_encoder.joblib")

prof = prof.drop(columns=["report_date"], errors="ignore")
prof.head()

,inn_id,ipul,id_region,main_okved_group,diff_datopen_report_date_flg
0,inn1000051,0,34,59,4
1,inn1000246,1,22,15,6
2,inn1000281,1,29,18,8
3,inn1000367,1,30,23,4
4,inn1000409,0,34,59,8


### Числовые признаки

Хотим в признаках зафиксировать связь предыдущих событий на будующие. Для этого будем использовать лаги.

In [22]:
def _lags_row_at_cutoff(n: int, weeks: np.ndarray, y: np.ndarray, max_lag: int = 70):
    """Одна строка лагов для недели n (эквивалент shift)"""

    mask = (weeks <= n) & (weeks >= n - max_lag)
    idx = np.flatnonzero(mask)

    w = weeks[idx] # отфильтрованные недели в нужном диапозоне 
    yy = y[idx] # отфильтрованные таргеты в нужном диапозоне
    pos = int(np.where(w == n)[0][0])

    row = {}
    for lag in range(1, max_lag + 1):
        j = pos - lag
        row[f"lag_{lag}"] = float(yy[j])

    return row

def get_features(n, df) -> pd.DataFrame:
    """
    n - номер недели
    df - датафрейм с транзакциями одного inn_id
    """

    max_lag = 70

    g = df.sort_values("week")
    weeks = g["week"].to_numpy()
    y = g["target"].to_numpy(dtype=float)
    row = _lags_row_at_cutoff(n, weeks, y, max_lag)
    
    return pd.DataFrame([row]) if row else pd.DataFrame()


In [23]:
def build_features_horizons_for_inn(
    n_target: int, k: int, inn_df: pd.DataFrame, max_lag: int = 70
) -> pd.DataFrame:
    """Все горизонты для одного ИНН """

    g = inn_df.sort_values("week")
    weeks = g["week"].to_numpy()
    y = g["target"].to_numpy(dtype=float)

    rows = []
    for i in range(k):
        cutoff = n_target - i
        row = _lags_row_at_cutoff(cutoff, weeks, y, max_lag)

        values = row.values()
        row['lag_max'] = max(values)
        row['lag_min'] = min(values)
        row['lag_mean'] = sum(values) / len(values)
        
        row['inn_id'] = inn_df['inn_id'].iloc[0] # чтобы потом по inn доклеить profiles 
        row['horizon'] = i + 1 
        rows.append(row) 
        
    return pd.DataFrame(rows) if rows else pd.DataFrame()

Представим, что мы хотим предсказать на одну неделю вперед. Оставим 106-117 недели для предсказания и сравнения скора с лидербордом. Значит, в трене можно использовать 0 - 105 недель. Тогда мы будем считать 105 неделю таргетом, а строить признаки на 0-104 неделях. 

Если мы хотим предсказать 2 неделю, таргетом снова считаем 105 неделю, но признаки уже строим на 0-103 неделях. 

Если мы хотим предсказать 3 неделю, мы строрим признаки на 0-102 неделях. 

Тогда чтобы предсказать неделю k надо строить признаки на неделях 105-k. Так мы будем показывать как влияют предыдущие значения на будующую неделю номер k. 

Предсказание на 12 недель: предсказание на 1, предсказание на 2, .... предсказание на 12 неделю.

In [ ]:
train = pd.DataFrame()
n = 105 # текущая неделя 
k = 12 # горизонт предсказания

train_list = []

df_week_n = trns.loc[trns["week"] == n, ["inn_id", "target"]]
inn_subset = df_week_n["inn_id"].unique()
inn_groups = {gid: g for gid, g in trns.groupby("inn_id", sort=False)}

train_list = []
for inn in inn_subset:
    inn_df = inn_groups.get(inn)

    t_series = df_week_n.loc[df_week_n["inn_id"] == inn, "target"]
    t_val = t_series.iloc[0]

    f_block = build_features_horizons_for_inn(n, k, inn_df)
    f_block["target"] = t_val
    train_list.append(f_block)
        
train = pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame() 

# добавление категориальных признаков 
train = train.merge(prof, on="inn_id", how="left")
for col in cat_cols:
    train[col] = train[col].fillna(-1).astype("category")
train = train.drop(columns=["inn_id"], errors="ignore")

# логарифмирование 
lag_target_cols = [c for c in train.columns if c.startswith("lag_") or c == "target" or c == 'trns_amount']
train[lag_target_cols] = train[lag_target_cols].applymap(lambda x: np.log(x + 1))

train.head()


In [98]:
del train
gc.collect()

0

Чтобы увеличить обучающую выборку, попробуем не фиксировать таргетную неделю на n = 105. Построим по схеме выше признаки для 10 таргенных недель (105...95). 

In [99]:
def sample_n(n, df) -> pd.DataFrame:
    """ Собрать обучающую выборку для недели n """
    k = 12 # горизонт предсказания

    df_week_n = df.loc[df["week"] == n, ["inn_id", "target"]]
    inn_subset = df_week_n["inn_id"].unique()
    inn_groups = {gid: g for gid, g in df.groupby("inn_id", sort=False)}

    train_list = []
    for inn in inn_subset:
        inn_df = inn_groups.get(inn)

        t_series = df_week_n.loc[df_week_n["inn_id"] == inn, "target"]
        t_val = t_series.iloc[0]

        f_block = build_features_horizons_for_inn(n, k, inn_df)
        f_block["target"] = t_val
        train_list.append(f_block)

    return pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame()

In [100]:
import time

start_time = time.perf_counter()

N = 10
train_parts = []

for i in range(N):
    n = 105 - i
    train_parts.append(sample_n(n, trns))

train = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame()

# добавление категориальных признаков 
train = train.merge(prof, on="inn_id", how="left")
for col in cat_cols:
    train[col] = train[col].fillna(-1).astype("category")
train = train.drop(columns=["inn_id"], errors="ignore")

# логарифмирование 
lag_target_cols = [c for c in train.columns if c.startswith("lag_") or c == "target" or c == 'trns_amount']
train[lag_target_cols] = train[lag_target_cols].applymap(lambda x: np.log(x + 1))

end_time = time.perf_counter()
execution_time = (end_time - start_time) / 60

print(f"Время выполнения: {execution_time:.2f} минут")

train.head()

Время выполнения: 20.13 минут


,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_70,lag_max,lag_min,lag_mean,horizon,target,ipul,id_region,main_okved_group,diff_datopen_report_date_flg
0,11.487807,14.610071,15.044252,11.130213,11.644807,10.373773,9.118413,13.542485,13.573879,14.297409,...,12.023371,15.044252,0.0,13.035779,1,15.821142,0,34,59,4
1,14.610071,15.044252,11.130213,11.644807,10.373773,9.118413,13.542485,13.573879,14.297409,11.258165,...,11.995143,15.044252,0.0,13.037730,2,15.821142,0,34,59,4
2,15.044252,11.130213,11.644807,10.373773,9.118413,13.542485,13.573879,14.297409,11.258165,9.807153,...,11.356684,15.044252,0.0,12.971237,3,15.821142,0,34,59,4
3,11.130213,11.644807,10.373773,9.118413,13.542485,13.573879,14.297409,11.258165,9.807153,11.321931,...,12.224679,14.972081,0.0,12.853014,4,15.821142,0,34,59,4
4,11.644807,10.373773,9.118413,13.542485,13.573879,14.297409,11.258165,9.807153,11.321931,12.904122,...,12.105032,14.972081,0.0,12.857099,5,15.821142,0,34,59,4


In [101]:
train.to_parquet('train_only_vtb.parquet')

In [ ]:
# с признаками по ряду переводов со счетов любого банка 
# train.to_parquet('train_extended.parquet')

## Обучение

In [102]:
!pip3 install xgboost

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [30]:
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error

In [39]:
def transform_data_for_prediction(df, prof, inns, n_start_week):
    inn_groups = {gid: g for gid, g in df.groupby("inn_id", sort=False)}
        
    res = []
    for inn in inns:
        df_inn = inn_groups.get(inn)

        g = df_inn.sort_values("week")
        weeks = g["week"].to_numpy()
        y = g["target"].to_numpy(dtype=float)

        row = _lags_row_at_cutoff(n_start_week, weeks, y)
        
        values = row.values()
        row['lag_max'] = max(values)
        row['lag_min'] = min(values)
        row['lag_mean'] = sum(values) / len(values)

        row['inn_id'] = inn

        res.append(row)
    
    res_df = pd.DataFrame(res)

    l = len(res_df)

    # res_df = res_df.loc[res_df.index.repeat(12)].copy()
    # res_df["horizon"] = list(range(1, 13)) * l

    res_df["horizon"] = 1

    # добавление категориальных признаков 
    res_df = res_df.merge(prof, on="inn_id", how="left")
    for col in cat_cols:
        res_df[col] = res_df[col].fillna(-1).astype("category")
    res_df = res_df.drop(columns=["inn_id"], errors="ignore")

    # логарифмирование 
    lag_target_cols = [c for c in res_df.columns if c.startswith("lag_") or c == "target" or c == 'trns_amount']
    res_df[lag_target_cols] = res_df[lag_target_cols].applymap(lambda x: np.log(x + 1))
    

    return res_df


In [40]:
ts = target_series[(target_series['week'] >= 106) & (target_series['week'] <= 106)].copy()
ts = ts.sort_values(by=['inn_id', 'week'])
y_test = ts['target']

trns = trns.sort_values(by=['inn_id', 'week'])
X_test = transform_data_for_prediction(trns, prof, ts['inn_id'], 106)

In [41]:
train = pd.read_parquet("train_only_vtb.parquet")

In [42]:
train = train.dropna()

y_train = train["target"]
X_train = train.drop(columns=["target"])

model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    enable_categorical=True,
)

start_time = time.perf_counter()

model.fit(X_train, y_train)

end_time = time.perf_counter()
execution_time = (end_time - start_time) / 60

print(f"Время выполнения: {execution_time:.2f} минут")

pred_log = model.predict(X_test)
pred = np.expm1(pred_log)

Время выполнения: 1.73 минут


In [44]:
X_test

,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_69,lag_70,lag_max,lag_min,lag_mean,horizon,ipul,id_region,main_okved_group,diff_datopen_report_date_flg
0,15.821142,11.487807,14.610071,15.044252,11.130213,11.644807,10.373773,9.118413,13.542485,13.573879,...,12.636438,13.423724,15.821142,0.000000,13.329652,1,0,34,59,4
1,15.626517,15.110887,15.627230,15.053898,15.524151,15.779968,15.219413,14.639067,14.987862,14.969566,...,14.188885,13.881140,16.092738,0.000000,14.695562,1,1,22,15,6
2,10.328082,10.774144,9.141981,12.785031,12.123938,12.815202,11.652865,11.490739,11.325688,13.429049,...,12.785570,12.667454,14.259361,0.000000,12.589239,1,1,29,18,8
3,14.022167,13.661212,12.272767,13.718080,13.343955,14.062136,13.394811,13.689302,13.887568,14.189779,...,13.286121,13.917111,15.027966,0.000000,13.770551,1,1,30,23,4
4,0.000000,13.015916,12.999253,13.293337,7.599457,11.949798,12.403199,9.961868,10.120735,11.971783,...,13.425210,0.000000,13.824686,0.000000,12.136759,1,0,34,59,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51958,15.368828,15.710406,15.393540,16.016178,15.961055,15.256314,15.697546,15.585403,15.705385,16.071476,...,15.269463,15.478880,16.296947,9.662077,15.580379,1,1,56,32,8
51959,12.198977,12.699588,12.938880,12.295488,12.524511,12.455830,12.569290,12.003098,12.254917,12.295468,...,12.259195,12.275113,13.402797,10.767151,12.383016,1,0,35,42,8
51960,0.000000,0.000000,0.000000,0.000000,15.371876,12.877789,13.456792,13.150640,13.269197,0.000000,...,11.938366,9.943552,16.277075,0.000000,14.768911,1,1,35,20,4
51961,16.255235,16.094133,16.202227,16.118030,15.891348,16.081322,16.312722,16.374579,16.790556,16.049659,...,15.852461,15.348424,16.889736,13.008213,16.076656,1,1,56,23,6


In [45]:
rmsle = np.sqrt(mean_squared_log_error(y_test, pred))
print(f"RMSLE: {rmsle}")

RMSLE: 2.105167617636681


In [46]:
pred

array([2.5624316e+05, 3.6861358e+06, 1.0677203e+05, ..., 1.2713899e+02,
       1.1617492e+07, 1.5773637e+07], shape=(51963,), dtype=float32)